# Statistical Language Semantics and Weighting

Today we move from raw co-occurence counts to information-theoretic weighting.

Raw counts are biased toward frequent words like "the" or "is".

This is fixed using:
- Pointwise Mutual Information (PMI)
- Positive PMI (PPMI)

This transforms a co-occurence matrix into a semantic weighting space.

### Objective

- Convert co-occurence counts to probabilities
- Compute PMI matrix
- Apply PPMI transformation
- Interpret semantic associations beyond raw frequency bias

### Core Constraint

- No sklearn / NLP libraries
- Only numpy and Python
- Must handle zero-frequency safely
- Vectorized operations preferred over loops

### Expected Output

```

STATISTICAL LANGUAGE SEMANTICS & WEIGHTING (v1)

CORPUS STATISTICS
Vocabulary Size: 24
Token Count: 28

CO-OCCURRENCE MATRIX
Shape: (24, 24)
Total Co-occurrences: 56

SEMANTIC ASSOCIATIONS
dense ↔ matrices: 2.5850
semantic ↔ meaning: 2.1203

```

### Imports

In [9]:
import numpy as np
from collections import Counter

### Corpus (Shared Setup)

In [10]:
corpus = (
    "the vector space model represents text as dense matrices "
    "dense matrices capture semantic meaning via spatial proximity "
    "text meaning can be aligned across different models using linear maps"
)

tokens = corpus.split()
vocab = sorted(set(tokens))
word_to_index = {w: i for i, w in enumerate(vocab)}
index_to_word = {i: w for w, i in word_to_index.items()}

### Co-Occurence Matrix (From day 7)

In [11]:
def build_cooccurence_matrix(tokens, word_to_index, window_size):
    """Construct raw co-occurence matrix."""

    vocab_size = len(word_to_index)
    matrix = np.zeros((vocab_size, vocab_size), dtype = np.int32)

    for center_index, center_word in enumerate(tokens):
        center_id = word_to_index[center_word]
        start = max(0, center_index - window_size)
        end = min(len(tokens), center_index + window_size + 1)

        for context_index in range(start, end):
            if context_index == center_index:
                continue

            context_word = tokens[context_index]
            context_id = word_to_index[context_word]
            
            matrix[center_id, context_id] += 1

    return matrix

### Probability Estimation Engine

In [12]:
def compute_probabilities(M):
    """Converts co-occurence matrix into probability distributions."""

    N = np.sum(M)

    if N == 0:
        return M, M, M, 0
    
    Pxy = M / N

    Px = np.sum(Pxy, axis=1, keepdims=True)
    Py = np.sum(Pxy, axis=0, keepdims=True)

    return Pxy, Px, Py, N

### PMI Compute Engine

In [13]:
def compute_pmi(Pxy, Px, Py):
    """PMI(i, j) = log2(P(i, j) / (P(i)P(j)))"""

    denominator = Px @ Py
    PMI = np.zeros_like(Pxy)
    mask = Pxy > 0
    ratio = np.zeros_like(Pxy)
    ratio[mask] = Pxy[mask] / denominator[mask]
    PMI[mask] = np.log2(ratio[mask])

    return PMI

### PMI Transformation Engine

In [14]:
def compute_ppmi(PMI):
    """Applies positive clipping to PMI matrix."""

    return np.maximum(0, PMI)

### Semantic Association Analysis

In [15]:
def analyze_pair(PPMI, word_to_index, w1, w2):
    """Extract semantic association strength."""

    i = word_to_index[w1]
    j = word_to_index[w2]

    return {"word_a": w1, "word_b": w2, "ppmi_score": PPMI[i, j]}

### Evaluation Harness

In [16]:
def evaluate_statistical_weighting(corpus, window_size=2):
    print("STATISTICAL LANGUAGE SEMANTICS & WEIGHTING (v1)\n")

    tokens = corpus.split()
    vocab = sorted(set(tokens))
    word_to_index = {w: i for i, w in enumerate(vocab)}

    M = build_cooccurence_matrix(tokens, word_to_index, window_size)

    Pxy, Px, Py, N = compute_probabilities(M)

    PMI = compute_pmi(Pxy, Px, Py)
    PPMI = compute_ppmi(PMI)

    print("CORPUS STATISTICS")
    print(f"Vocabulary Size: {len(vocab)}")
    print(f"Token Count: {len(tokens)}\n")

    print("CO-OCCURRENCE MATRIX")
    print(f"Shape: {M.shape}")
    print(f"Total Co-occurrences: {N}\n")

    print("SEMANTIC ASSOCIATIONS")

    pair1 = analyze_pair(PPMI, word_to_index, "dense", "matrices")
    pair2 = analyze_pair(PPMI, word_to_index, "semantic", "meaning")

    print(
        f"{pair1['word_a']} ↔ {pair1['word_b']}: "
        f"{pair1['ppmi_score']:.4f}"
    )

    print(
        f"{pair2['word_a']} ↔ {pair2['word_b']}: "
        f"{pair2['ppmi_score']:.4f}"
    )

### Execute Pipeline

In [17]:
evaluate_statistical_weighting(corpus, window_size=2)

STATISTICAL LANGUAGE SEMANTICS & WEIGHTING (v1)

CORPUS STATISTICS
Vocabulary Size: 24
Token Count: 28

CO-OCCURRENCE MATRIX
Shape: (24, 24)
Total Co-occurrences: 106

SEMANTIC ASSOCIATIONS
dense ↔ matrices: 2.3129
semantic ↔ meaning: 1.7279
